# Productive capacity: capital formation scenarios

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zistgah/hecon/blob/main/notebooks/01_capital_formation_scenarios.ipynb)

The paper's three forms of capital, run as a scenario model: learner clinical-reasoning capital, institutional educational-process capital and shared digital-infrastructure capital, compared across conventional teaching (comparator), AI-assisted teaching that only harvests existing tools (harvest), and AI-assisted teaching that also contributes back (cultivate).

**How to use it.** Press ▶ on each step in order, or choose Runtime, then Run all. Each step's code is hidden behind its title; double-click the title to see it. Educational use only: nothing here is a medical device or a finding.

© 1993–2026 Abhishek Choudhary. All rights reserved. AyeAI. Code GPL-3.0-or-later; text CC BY-SA 4.0. AyeAI® and AyeAM® are registered trade marks of Abhishek Choudhary; AyeCNSe™ (Indian TM application No. 7415871, pending). Cite with [CITATION.cff](https://github.com/zistgah/hecon/blob/main/CITATION.cff); the DOI is recorded in the repository README once minted.

## The model, in the paper's terms

Learner capability accumulates and depreciates (paper, section 3.3):

$$K_{i,t+1} = (1-\delta)K_{i,t} + \varphi(P_{i,t}, F_{i,t}, E_{i,t})$$

Practice $P$ and feedback $F$ rise with AI only to the extent that its questions and feedback are aligned with defined competencies (the *validity* setting). Institutional and shared capital follow $X_{t+1} = (1-\Delta)X_t + J_t$ (section 10.4), and productive capacity is shown in the illustrative form $P_t = H_t^{a}(1+I_t)^{b}(1+D_t)^{c}$ (section 10.3). Every number is an assumption you set; the defaults only make the mechanics visible. They are not estimates.

In [ ]:
#@title Set up: load the hecon package from the repository { display-mode: "form" }
import subprocess, sys
REPO = "zistgah/hecon"  #@param {type:"string"}
try:
    import hecon
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/%s@main" % REPO], check=True)
    import hecon
print(hecon.NOTICE)

## Step 1: set your assumptions

*validity* is how well AI questions and feedback map to defined competencies and validated measures (0 none, 1 complete). *capture* switches structured reasoning records on (1) or off (0); records are what build institutional capital. The costs are in any currency you choose, per year.

In [ ]:
#@title Your assumptions { display-mode: "form" }
validity = 0.6  #@param {type:"slider", min:0, max:1, step:0.05}
ai_practice_mult = 1.6  #@param {type:"slider", min:1, max:3, step:0.1}
ai_feedback_mult = 1.5  #@param {type:"slider", min:1, max:3, step:0.1}
capture = 1.0  #@param {type:"slider", min:0, max:1, step:1}
cohort_size = 150  #@param {type:"integer"}
years = 10  #@param {type:"integer"}
tech_fixed = 20000  #@param {type:"number"}
governance_fixed = 15000  #@param {type:"number"}
cultivation_cost = 12000  #@param {type:"number"}
from hecon import model
P = dict(validity=validity, ai_practice_mult=ai_practice_mult, ai_feedback_mult=ai_feedback_mult, capture=capture,
         cohort_size=cohort_size, years=years, tech_fixed=tech_fixed, governance_fixed=governance_fixed, cultivation_cost=cultivation_cost)
R = model.simulate(P)
print(model.report(R))

## Step 2: read the trajectories

The first chart counts learners who reach the competency threshold each year. The second shows the capital stocks behind them. A higher count is not by itself a reason to adopt: it has to be weighed against cost, which notebook 02 does.

In [ ]:
#@title Draw the trajectories { display-mode: "form" }
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for arm in model.ARMS:
    ax[0].plot(R["years"], R["arms"][arm]["competent"], marker="o", label=arm)
    ax[1].plot(R["years"], R["arms"][arm]["capacity"], label=arm)
ax[0].set_title("Learners reaching the threshold, per cohort"); ax[0].set_xlabel("year"); ax[0].legend()
ax[1].set_title("Productive capacity index (illustrative)"); ax[1].set_xlabel("year"); ax[1].legend()
plt.tight_layout(); plt.show()

## Step 3: AI is an input, not the outcome (Proposition 2)

With validity at zero the AI arms produce more activity and no extra competence. The sweep below shows how the gain depends on validity alone, with everything else held at your settings.

In [ ]:
#@title Sweep validity from 0 to 1 { display-mode: "form" }
vals = [i / 10 for i in range(11)]
gain = [sum(model.simulate(dict(P, validity=v))["arms"]["harvest"]["competent"]) - sum(R["arms"]["comparator"]["competent"]) for v in vals]
plt.figure(figsize=(6, 3.5)); plt.plot(vals, gain, marker="o")
plt.title("Extra competent learners over the horizon"); plt.xlabel("validity"); plt.axhline(0, color="grey", lw=0.8); plt.show()
print("At validity 0 the extra competent learners are", gain[0])

## Step 4: harvesting and cultivating

Cultivation costs money every year. What it buys is shared infrastructure $D$, which lowers the cost of the next deployment for this institution and for others. The chart shows both stocks.

In [ ]:
#@title Shared infrastructure and institutional capital { display-mode: "form" }
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for arm in ("harvest", "cultivate"):
    ax[0].plot(R["years"], R["arms"][arm]["D"], label=arm); ax[1].plot(R["years"], R["arms"][arm]["I"], label=arm)
ax[0].set_title("Shared digital-infrastructure capital D"); ax[1].set_title("Institutional process capital I")
for a in ax: a.set_xlabel("year"); a.legend()
plt.tight_layout(); plt.show()

## What this does not show

The functional forms and parameters need empirical estimation (paper, section 14). Use this notebook to make assumptions explicit and to see which ones decide the answer, then measure those in a study (section 12).